# 1.5 内存、调度与扩展性

## 本节学习目标

- 分析 buffer reuse 和 Host/Device 搬移与同步
- 比较 partition 与 rank scaling

## 环境检查

直接检查本节需要的运行环境；若检查失败，请先在对应 CPU/NPU 节点加载课程要求的工具链。


In [ ]:
%%bash
set -e
command -v cmake
command -v msprof
command -v npu-smi
npu-smi info
printf "ASCEND_HOME_PATH=%s\n" "${ASCEND_HOME_PATH:?请先 source CANN set_env.sh}"


## 内存与同步

工程复用 CSR 分片、Arnoldi buffer 和 HCCL Device buffer，避免迭代区重复分配。正式路径中 CSR 与 Krylov 向量在 solve 内常驻 Device，SpMV/Dot/Norm/AXPY/Scale 由 Ascend C RTC kernel 执行，HCCL 直接读写 Device buffer；剩余 transfer 与同步临界路径主要来自首尾搬移和 HCCL 同步。

## 分区与 scaling

rows 分区是负载不均 baseline，nnz 分区更贴近 SpMV 工作量。rank 增加同时改变局部计算、collective、线程预算和同步等待，必须结合全字段分析。

## 预期现象与结果分析

优化目标应是端到端临界路径，不是单独压低某个字段。当前 SpMV/Dot/Norm/AXPY/Scale 已是 Ascend C RTC Device kernel；性能结论必须以当前真机实测为依据，不引用历史 reference 路径数据。

## 课后实践

比较 rows/nnz 分区或不同 rank 数，写出需要同时控制的变量。

参考答案见 `answer/01.05_answer.md`。